# Identificação do projeto

**Título do projeto:** Carro, piloto ou estratégia? A Fórmula 1 na era híbrida, de 2014 a 2025
**Integrantes do grupo:** Zhou Ka Shen, Lucas Gabriel, Vinicius Wamser e Fabio Henrique
**Turma:** Bacharelado em Sistemas de Informação, 6º período, Data Analytics, PUCPR
**Descrição breve do problema ou contexto:** Na Fórmula 1, o resultado de uma corrida depende do carro, do piloto, da estratégia da equipe e de o carro quebrar ou não. Queremos preparar os dados das temporadas de 2014 a 2025 para medir o peso de cada um desses fatores e ver se a mudança de regras de 2022 alterou esse equilíbrio.

## Configuração

Este notebook foi feito para rodar no Google Colab. A primeira célula instala as duas bibliotecas que o Colab não traz, o PySpark e o kagglehub. Depois carregamos as bibliotecas do RA1 e criamos as pastas onde os dados ficam guardados, que no Colab ficam dentro de /content/data. A semente 42 faz qualquer sorteio sair sempre igual.

In [1]:
# Instalação no Google Colab
import subprocess
import sys

NO_COLAB = "google.colab" in sys.modules
if NO_COLAB:
    pacotes = ["pyspark==3.5.3", "kagglehub==1.0.2"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pacotes], check=True)
print("Rodando no Colab:", NO_COLAB)

Rodando no Colab: True


In [2]:
# Configuração
import shutil
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd

np.random.seed(42)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.2.3 | numpy 2.1.3


In [3]:
# Configuração das pastas
PASTA_PROJETO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_RAW = PASTA_PROJETO / "data" / "raw"
PASTA_PROCESSED = PASTA_PROJETO / "data" / "processed"
PASTA_STREAM = PASTA_PROJETO / "data" / "stream"
for pasta in [PASTA_RAW, PASTA_PROCESSED, PASTA_STREAM]:
    pasta.mkdir(parents=True, exist_ok=True)
print("Pasta do projeto:", PASTA_PROJETO.name)

Pasta do projeto: content


---

# RA1 — Data Ingestion, Cleaning and Initial Analysis

Nesta etapa, carregue, prepare e explore os dados do projeto.

## 1. Contexto e objetivo

Apresente brevemente o problema investigado e o que o grupo pretende descobrir com os dados.

**Resposta do grupo:**

A Fórmula 1 é uma competição em que o resultado depende de várias coisas ao mesmo tempo: a qualidade do carro, a habilidade do piloto, a estratégia da equipe, como a hora certa de parar para trocar pneus, e a confiabilidade, ou seja, se o carro quebra ou não. Torcedores e analistas discutem o tempo todo quanto de uma vitória vem do carro e quanto vem do piloto.

Escolhemos as temporadas de 2014 a 2025, conhecidas como era híbrida. Desde 2014 os carros usam o mesmo tipo de motor, que combina combustível com energia elétrica. Em 2022 as regras dos carros mudaram bastante para facilitar as ultrapassagens, e isso nos permite comparar dois períodos: de 2014 a 2021 e de 2022 a 2025.

**Problema:** na era híbrida da Fórmula 1, quanto do resultado de uma corrida depende do carro, do piloto, da estratégia e da confiabilidade, e a mudança de regras de 2022 alterou esse equilíbrio?

**Objetivo do RA1:** preparar uma base de dados confiável para responder às perguntas abaixo nos próximos RAs. Neste RA as perguntas são apenas definidas.

**Recorte:**
- Temporadas de 2014 a 2025, todas completas. A temporada de 2026 fica de fora porque ainda está acontecendo.
- A unidade principal de análise é o resultado de um piloto em uma corrida.
- Usamos só as corridas principais. As corridas curtas de sábado, chamadas sprint, ficam de fora.

| ID | Pergunta | Tabelas usadas |
|---|---|---|
| P1 | Em quais circuitos a posição de largada mais define a posição final, e essa influência diminuiu depois das regras de 2022? | results, races, circuits |
| P2 | Quais pilotos superam o companheiro de equipe com frequência, tanto na classificação quanto na corrida? | qualifying, results, drivers, constructors |
| P3 | A estratégia de paradas, ou seja, quantas vezes o piloto para e em que volta faz a primeira parada, tem relação com ganhar ou perder posições em relação à largada? | pit_stops, results, races |
| P4 | Como a taxa de abandonos por quebra do carro mudou em cada equipe, e as equipes que mais quebraram perderam posições no campeonato por isso? | results, status, constructors |
| P5 | Quanto o piloto fica mais lento a cada volta conforme o pneu se desgasta, e isso muda de um circuito para outro? | lap_times, pit_stops, races, circuits |

**Dá para responder com esses dados?**
- P1, P3 e P4: a tabela results traz a posição de largada, a posição final e o motivo do fim de corrida de todos os pilotos.
- P2: a tabela qualifying guarda a classificação, a sessão que define a ordem de largada. Cada equipe tem dois pilotos, e formamos as duplas corrida a corrida, porque às vezes um piloto é trocado no meio do ano.
- P3: a tabela pit_stops registra as paradas nos boxes desde 2011.
- P5: a tabela lap_times tem o tempo de cada volta desde 1996.
- O que falta: não há dados sobre o tipo de pneu, o clima ou os momentos em que o carro de segurança entrou na pista. Também não existe uma divisão oficial entre quebra do carro e acidente, então vamos criar esse agrupamento.

## 2. Fonte e ingestão dos dados

- Identifique a fonte dos dados, incluindo link ou referência de acesso.
- Descreva brevemente o dataset e o que seus registros representam.
- Explique como os dados foram carregados e implemente a ingestão.

**Resposta do grupo:**

Usamos o conjunto de dados público Formula 1 World Championship 1950-2026, publicado por Atharv Ranjan no Kaggle: https://www.kaggle.com/datasets/atharvranjan/formula-1-world-championship-1950-present. A licença é livre e o download não exige login.

Os dados vêm em vários arquivos CSV, cada um com um tipo de informação: corridas, circuitos, pilotos, equipes, resultados, classificação, paradas nos boxes e voltas. Os arquivos se ligam por códigos numéricos, como raceId para a corrida e driverId para o piloto. Quando um valor não existe, o arquivo traz o texto \N.

A ingestão é feita toda pelo notebook: baixamos a versão 9 com a biblioteca kagglehub, copiamos os 9 arquivos que usamos para a pasta data/raw e lemos cada um com o pandas.

### 2.1 Download do dataset

Baixamos os dados com a biblioteca kagglehub. Como o conjunto é atualizado a cada corrida, fixamos a versão 9 no endereço do download. Assim, quem rodar o notebook depois recebe exatamente os mesmos dados.

In [4]:
# 2.1 Download do dataset
DATASET = "atharvranjan/formula-1-world-championship-1950-present/versions/9"
TABELAS = ["races", "circuits", "results", "drivers", "constructors", "status", "qualifying", "pit_stops", "lap_times"]

caminho_download = Path(kagglehub.dataset_download(DATASET))
print("Versão baixada:", caminho_download.name)

Using Colab cache for faster access to the 'formula-1-world-championship-1950-present' dataset.
Versão baixada: formula-1-world-championship-1950-present


O conjunto traz 17 arquivos, mas copiamos para a pasta data/raw só os 9 que vamos usar.

In [5]:
# 2.1 Cópia dos arquivos para data/raw
for nome in TABELAS:
    shutil.copy(caminho_download / f"{nome}.csv", PASTA_RAW / f"{nome}.csv")

arquivos_raw = sorted(arquivo.name for arquivo in PASTA_RAW.glob("*.csv"))
print(len(arquivos_raw), "arquivos em data/raw:", arquivos_raw)

9 arquivos em data/raw: ['circuits.csv', 'constructors.csv', 'drivers.csv', 'lap_times.csv', 'pit_stops.csv', 'qualifying.csv', 'races.csv', 'results.csv', 'status.csv']


### 2.2 Leitura das tabelas

Lemos as 9 tabelas do mesmo jeito, avisando ao pandas que o texto \N significa valor ausente. Depois damos a cada tabela um nome em português para facilitar a leitura do código.

In [6]:
# 2.2 Leitura das tabelas
tabelas = {}
for nome in TABELAS:
    tabelas[nome] = pd.read_csv(PASTA_RAW / f"{nome}.csv", na_values="\\N", encoding="utf-8")

corridas = tabelas["races"]
circuitos = tabelas["circuits"]
resultados = tabelas["results"]
pilotos = tabelas["drivers"]
equipes = tabelas["constructors"]
status = tabelas["status"]
classificacao = tabelas["qualifying"]
pit_stops = tabelas["pit_stops"]
voltas = tabelas["lap_times"]
print("Tabelas carregadas:", list(tabelas.keys()))

Tabelas carregadas: ['races', 'circuits', 'results', 'drivers', 'constructors', 'status', 'qualifying', 'pit_stops', 'lap_times']


### 2.3 Conferência do carregamento

Para conferir a leitura, montamos um resumo com o número de linhas e colunas de cada tabela e os anos que ela cobre.

In [7]:
# 2.3 Resumo das tabelas carregadas
ano_da_corrida = corridas.set_index("raceId")["year"]
linhas_resumo = []
for nome, tabela in tabelas.items():
    anos = "-"
    if "raceId" in tabela.columns:
        anos_tabela = tabela["raceId"].map(ano_da_corrida)
        anos = f"{anos_tabela.min():.0f}–{anos_tabela.max():.0f}"
    linhas_resumo.append({"tabela": nome, "linhas": len(tabela), "colunas": tabela.shape[1], "anos": anos})
resumo_ingestao = pd.DataFrame(linhas_resumo)
resumo_ingestao

,tabela,linhas,colunas,anos
0,races,1164,18,1950–2026
1,circuits,78,9,-
2,results,27568,18,1950–2026
3,drivers,879,9,-
4,constructors,214,5,-
5,status,141,2,-
6,qualifying,11298,9,1994–2026
7,pit_stops,12783,7,2011–2026
8,lap_times,632937,6,1996–2026


Também conferimos se todas as colunas que vamos usar existem em cada arquivo.

In [8]:
# 2.3 Conferência do esquema
COLUNAS_ESPERADAS = {
    "races": ["raceId", "year", "round", "circuitId", "name", "date"],
    "circuits": ["circuitId", "name", "location", "country"],
    "results": ["resultId", "raceId", "driverId", "constructorId", "grid", "position", "positionText", "positionOrder", "points", "laps", "time", "milliseconds", "statusId"],
    "drivers": ["driverId", "forename", "surname", "code", "dob", "nationality"],
    "constructors": ["constructorId", "name", "nationality"],
    "status": ["statusId", "status"],
    "qualifying": ["qualifyId", "raceId", "driverId", "constructorId", "position", "q1", "q2", "q3"],
    "pit_stops": ["raceId", "driverId", "stop", "lap", "duration", "milliseconds"],
    "lap_times": ["raceId", "driverId", "lap", "position", "time", "milliseconds"],
}
for nome, colunas in COLUNAS_ESPERADAS.items():
    faltando = set(colunas) - set(tabelas[nome].columns)
    print(f"{nome:13} colunas faltando: {len(faltando)}")

races         colunas faltando: 0
circuits      colunas faltando: 0
results       colunas faltando: 0
drivers       colunas faltando: 0
constructors  colunas faltando: 0
status        colunas faltando: 0
qualifying    colunas faltando: 0
pit_stops     colunas faltando: 0
lap_times     colunas faltando: 0


**Resultado:** a versão 9 foi baixada e as 9 tabelas foram lidas sem nenhuma coluna faltando. A tabela de resultados tem 27.568 linhas, de 1950 a 2026, e a de voltas tem 632.937 linhas, de 1996 a 2026. As tabelas de classificação, paradas e voltas começam antes de 2014, então cobrem todo o período escolhido. Nenhuma etapa foi feita à mão.

## 3. Exploração inicial dos dados

Investigue a quantidade de registros e colunas, os tipos de dados, os valores ausentes e outras características relevantes da base.

Nesta etapa só medimos e mostramos os problemas, sem alterar nenhum dado. O tratamento vem na seção 4. Cada verificação recebeu um código, de Q1 a Q14, para ser citada depois. Em alguns casos medimos tanto na base inteira, de 1950 a 2026, quanto só no período de 2014 a 2025.

### 3.1 Estrutura e significado de cada linha

| Tabela | Cada linha representa | Identificador |
|---|---|---|
| races | uma corrida | raceId |
| circuits | um circuito | circuitId |
| results | o resultado de um piloto em uma corrida | resultId, ou raceId com driverId |
| drivers | um piloto | driverId |
| constructors | uma equipe | constructorId |
| status | uma forma de terminar a corrida | statusId |
| qualifying | um piloto na classificação de uma corrida | qualifyId, ou raceId com driverId |
| pit_stops | uma parada de um piloto nos boxes | raceId com driverId e stop |
| lap_times | uma volta de um piloto | raceId com driverId e lap |

A tabela principal é a de resultados. Abaixo vemos suas colunas, o tipo de cada uma e quantos valores estão preenchidos.

In [9]:
# 3.1 Estrutura da tabela de resultados
resultados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27568 entries, 0 to 27567
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         27568 non-null  int64  
 1   raceId           27568 non-null  int64  
 2   driverId         27568 non-null  int64  
 3   constructorId    27568 non-null  int64  
 4   number           27562 non-null  float64
 5   grid             27568 non-null  int64  
 6   position         16494 non-null  float64
 7   positionText     27568 non-null  object 
 8   positionOrder    27568 non-null  int64  
 9   points           27568 non-null  float64
 10  laps             27568 non-null  int64  
 11  time             8356 non-null   object 
 12  milliseconds     8368 non-null   float64
 13  fastestLap       9026 non-null   float64
 14  rank             9284 non-null   float64
 15  fastestLapTime   9026 non-null   object 
 16  fastestLapSpeed  8252 non-null   float64
 17  statusId    

Vemos as primeiras linhas das tabelas principais para entender o que cada uma guarda.

In [10]:
# 3.1 Primeiras linhas das tabelas principais
display(resultados.head(3))
display(classificacao.head(3))
display(pit_stops.head(3))
display(voltas.head(3))

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22.0,1,1.0,1,1,10.0,58,1:34:50.616,5690616.0,39.0,2.0,1:27.452,218.300,1
1,2,18,2,2,3.0,5,2.0,2,2,8.0,58,+5.478,5696094.0,41.0,3.0,1:27.739,217.586,1
2,3,18,3,3,7.0,7,3.0,3,3,6.0,58,+8.163,5698779.0,41.0,5.0,1:28.090,216.719,1


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
0,1,18,1,1,22,1,1:26.572,1:25.187,1:26.714
1,2,18,9,2,4,2,1:26.103,1:25.315,1:26.869
2,3,18,5,1,23,3,1:25.664,1:25.452,1:27.079


,raceId,driverId,stop,lap,time,duration,milliseconds
0,841,153,1,1,17:05:23,26.898,26898
1,841,30,1,1,17:05:52,25.021,25021
2,841,17,1,11,17:20:48,23.426,23426


,raceId,driverId,lap,position,time,milliseconds
0,841,20,1,1,1:38.109,98109
1,841,20,2,1,1:33.006,93006
2,841,20,3,1,1:32.713,92713


Montamos um dicionário de dados só com as colunas que vamos usar. Ele mostra o que cada coluna significa e o tipo com que o pandas a leu.

In [11]:
# 3.1 Dicionário de dados (parte 1)
itens_dicionario = [
    ["results", "raceId", "identificador da corrida", "inteiro (chave)"],
    ["results", "driverId", "identificador do piloto", "inteiro (chave)"],
    ["results", "constructorId", "identificador da equipe", "inteiro (chave)"],
    ["results", "grid", "posição de largada", "0 = largou do pit lane; 1, 2, 3..."],
    ["results", "position", "posição final oficial", "1, 2, 3... ou vazio se não classificado"],
    ["results", "positionText", "posição final em texto", "número ou código (R, D, W, N, E, F)"],
    ["results", "positionOrder", "ordem final para ordenar todos", "1, 2, 3..."],
    ["results", "points", "pontos no campeonato", "pontos"],
    ["results", "laps", "voltas completadas", "voltas"],
    ["results", "statusId", "identificador do status final", "inteiro (chave)"],
    ["status", "status", "como o piloto terminou a corrida", "texto (Finished, +1 Lap, Engine...)"],
    ["races", "year", "temporada", "ano"],
    ["races", "round", "etapa dentro da temporada", "1, 2, 3..."],
    ["races", "date", "data da corrida", "AAAA-MM-DD"],
]
len(itens_dicionario)

14

In [12]:
# 3.1 Dicionário de dados (parte 2)
itens_dicionario = itens_dicionario + [
    ["qualifying", "position", "posição na classificação", "1, 2, 3..."],
    ["qualifying", "q1", "melhor tempo no Q1", "texto m:ss.sss"],
    ["qualifying", "q2", "melhor tempo no Q2", "texto m:ss.sss ou vazio"],
    ["qualifying", "q3", "melhor tempo no Q3", "texto m:ss.sss ou vazio"],
    ["pit_stops", "stop", "número da parada do piloto", "1, 2, 3..."],
    ["pit_stops", "lap", "volta em que a parada ocorreu", "volta"],
    ["pit_stops", "milliseconds", "duração da parada (entrada a saída do pit lane)", "milissegundos"],
    ["lap_times", "lap", "número da volta", "volta"],
    ["lap_times", "position", "posição do piloto naquela volta", "1, 2, 3..."],
    ["lap_times", "milliseconds", "tempo da volta", "milissegundos"],
]
dicionario = pd.DataFrame(itens_dicionario, columns=["tabela", "atributo", "significado", "unidade_dominio"])
tipos_observados = []
for nome, atributo in zip(dicionario["tabela"], dicionario["atributo"]):
    tipos_observados.append(str(tabelas[nome][atributo].dtype))
dicionario.insert(2, "tipo_observado", tipos_observados)
dicionario

,tabela,atributo,tipo_observado,significado,unidade_dominio
0,results,raceId,int64,identificador da corrida,inteiro (chave)
1,results,driverId,int64,identificador do piloto,inteiro (chave)
2,results,constructorId,int64,identificador da equipe,inteiro (chave)
3,results,grid,int64,posição de largada,"0 = largou do pit lane; 1, 2, 3..."
4,results,position,float64,posição final oficial,"1, 2, 3... ou vazio se não classificado"
5,results,positionText,object,posição final em texto,"número ou código (R, D, W, N, E, F)"
6,results,positionOrder,int64,ordem final para ordenar todos,"1, 2, 3..."
7,results,points,float64,pontos no campeonato,pontos
8,results,laps,int64,voltas completadas,voltas
9,results,statusId,int64,identificador do status final,inteiro (chave)


### 3.2 Valores ausentes: Q1

Criamos uma função que conta quantos valores faltam em cada coluna e qual o percentual sobre o total de linhas. Primeiro aplicamos a todas as tabelas, com todos os anos.

In [13]:
# 3.2 Valores ausentes nos dados brutos
def contar_nulos(tabela):
    nulos = tabela.isna().sum()
    nulos = nulos[nulos > 0]
    percentual = (nulos / len(tabela) * 100).round(1)
    return pd.DataFrame({"nulos": nulos, "percentual": percentual})

partes = []
for nome, tabela in tabelas.items():
    partes.append(contar_nulos(tabela).assign(tabela=nome))
nulos_brutos = pd.concat(partes)
nulos_brutos

,nulos,percentual,tabela
time,731,62.8,races
fp1_date,1035,88.9,races
fp1_time,1057,90.8,races
fp2_date,1046,89.9,races
fp2_time,1068,91.8,races
fp3_date,1064,91.4,races
fp3_time,1083,93.0,races
quali_date,1035,88.9,races
quali_time,1057,90.8,races
sprint_date,1135,97.5,races


Agora olhamos só o período de 2014 a 2025, nas duas tabelas mais importantes: resultados e classificação.

In [14]:
# 3.2 Valores ausentes no recorte 2014–2025
ids_2014_2025 = corridas.loc[corridas["year"].between(2014, 2025), "raceId"]
resultados_2014_2025 = resultados[resultados["raceId"].isin(ids_2014_2025)]
classificacao_2014_2025 = classificacao[classificacao["raceId"].isin(ids_2014_2025)]
print("Linhas de resultados no recorte:", len(resultados_2014_2025))
display(contar_nulos(resultados_2014_2025))
print("Linhas de classificação no recorte:", len(classificacao_2014_2025))
display(contar_nulos(classificacao_2014_2025))

Linhas de resultados no recorte: 5105


,nulos,percentual
position,776,15.2
time,2279,44.6
milliseconds,2274,44.5
fastestLap,239,4.7
rank,22,0.4
fastestLapTime,239,4.7
fastestLapSpeed,696,13.6


Linhas de classificação no recorte: 5089


,nulos,percentual
q1,78,1.5
q2,1364,26.8
q3,2650,52.1


**Resultado:** no período escolhido, 776 dos 5.105 resultados, ou 15,2%, não têm posição final. São pilotos que abandonaram a corrida ou não foram classificados. A classificação acontece em três fases eliminatórias, chamadas Q1, Q2 e Q3. Por isso 1.364 pilotos, ou 26,8%, não têm tempo no Q2 e 2.650, ou 52,1%, não têm tempo no Q3: eles foram eliminados antes. Outros 78 não têm tempo no Q1. Esses vazios fazem sentido e não são erro de coleta. Na base inteira, a posição final está vazia em 40,2% dos resultados, e há 1 parada sem duração em texto.

### 3.3 Registros duplicados: Q2

Verificamos se algum registro aparece repetido. Por exemplo, o mesmo piloto não pode ter dois resultados na mesma corrida. Também procuramos linhas totalmente iguais.

In [15]:
# 3.3 Duplicatas pela chave natural
CHAVES = [
    ("races", ["raceId"]), ("races", ["year", "round"]), ("circuits", ["circuitId"]),
    ("results", ["raceId", "driverId"]), ("drivers", ["driverId"]), ("constructors", ["constructorId"]),
    ("status", ["statusId"]), ("qualifying", ["raceId", "driverId"]),
    ("pit_stops", ["raceId", "driverId", "stop"]), ("lap_times", ["raceId", "driverId", "lap"]),
]
linhas_duplicatas = []
for nome, chave in CHAVES:
    tabela = tabelas[nome]
    recorte = tabela[tabela["raceId"].isin(ids_2014_2025)] if "raceId" in tabela.columns else tabela
    linhas_duplicatas.append({
        "tabela": nome,
        "chave": " + ".join(chave),
        "duplicatas_brutos": tabela.duplicated(subset=chave).sum(),
        "duplicatas_2014_2025": recorte.duplicated(subset=chave).sum(),
        "linhas_identicas": tabela.duplicated().sum(),
    })
duplicatas = pd.DataFrame(linhas_duplicatas)
duplicatas

,tabela,chave,duplicatas_brutos,duplicatas_2014_2025,linhas_identicas
0,races,raceId,0,0,0
1,races,year + round,0,0,0
2,circuits,circuitId,0,0,0
3,results,raceId + driverId,91,0,0
4,drivers,driverId,0,0,0
5,constructors,constructorId,0,0,0
6,status,statusId,0,0,0
7,qualifying,raceId + driverId,0,0,0
8,pit_stops,raceId + driverId + stop,0,0,0
9,lap_times,raceId + driverId + lap,0,0,0


**Resultado:** só a tabela de resultados tem repetições: 91 casos de um mesmo piloto duas vezes na mesma corrida, todos entre 1950 e 1978. Naquela época dois pilotos podiam dividir o mesmo carro, então o registro é real. No período de 2014 a 2025 não há nenhuma repetição, e nenhuma tabela tem linhas totalmente iguais.

### 3.4 Tipos de dados: Q3, Q4 e Q14

Conferimos como o pandas leu as colunas de data, tempo e posição. Um tempo guardado como texto não permite fazer contas, como calcular uma média.

In [16]:
# 3.4 Tipos das colunas de data, tempo e posição (Q3)
COLUNAS_TIPO = [("races", "date"), ("results", "position"), ("results", "time"), ("results", "fastestLapTime"),
                ("qualifying", "q1"), ("qualifying", "q2"), ("qualifying", "q3"),
                ("pit_stops", "duration"), ("lap_times", "time"), ("lap_times", "milliseconds")]
linhas_tipo = []
for nome, coluna in COLUNAS_TIPO:
    exemplo = tabelas[nome][coluna].dropna().iloc[0]
    linhas_tipo.append({"tabela": nome, "coluna": coluna, "tipo_lido": str(tabelas[nome][coluna].dtype), "exemplo": exemplo})
tipos_lidos = pd.DataFrame(linhas_tipo)
tipos_lidos

,tabela,coluna,tipo_lido,exemplo
0,races,date,object,2009-03-29
1,results,position,float64,1.0
2,results,time,object,1:34:50.616
3,results,fastestLapTime,object,1:27.452
4,qualifying,q1,object,1:26.572
5,qualifying,q2,object,1:25.187
6,qualifying,q3,object,1:26.714
7,pit_stops,duration,object,26.898
8,lap_times,time,object,1:38.109
9,lap_times,milliseconds,int64,98109


Os tempos aparecem em formatos diferentes. Para contar quantos formatos existem, trocamos cada dígito por 9. Assim, 1:27.452 vira 9:99.999, e todos os tempos com o mesmo formato ficam iguais.

In [17]:
# 3.4 Formatos de tempo encontrados (Q4)
COLUNAS_TEMPO = [("results", "time"), ("qualifying", "q1"), ("pit_stops", "duration"), ("lap_times", "time")]
for nome, coluna in COLUNAS_TEMPO:
    valores = tabelas[nome][coluna].dropna().astype(str)
    formatos = valores.str.replace(r"\d", "9", regex=True)
    print(f"{nome}.{coluna}: {formatos.nunique()} formatos ->", formatos.value_counts().head(6).to_dict())

results.time: 21 formatos -> {'+99.999': 3342, '+9:99.999': 1771, '+9.999': 843, '9:99:99.999': 804, '+9:99.99': 281, '+9:99.9': 279}
qualifying.q1: 2 formatos -> {'9:99.999': 11113, '99:99.999': 1}
pit_stops.duration: 3 formatos -> {'99.999': 12195, '99:99.999': 521, '9:99.999': 66}
lap_times.time: 3 formatos -> {'9:99.999': 632251, '99:99.999': 643, '9:99:99.999': 43}


Criamos a função tempo_para_segundos, que transforma um texto como 1:27.452 em 87,452 segundos. Ela separa o texto nos dois-pontos e vai multiplicando por 60. Usamos essa função no diagnóstico e na limpeza.

In [18]:
# 3.4 Função de conversão de tempo
def tempo_para_segundos(texto):
    if pd.isna(texto):
        return np.nan
    segundos = 0.0
    for parte in str(texto).split(":"):
        segundos = segundos * 60 + float(parte)
    return segundos

print(tempo_para_segundos("1:27.452"), tempo_para_segundos("16:44.718"), tempo_para_segundos("1:34:50.616"))

87.452 1004.718 5690.616


Algumas tabelas guardam o mesmo tempo duas vezes, em texto e em milissegundos. Conferimos se os dois valores batem, aceitando até 1 milissegundo de diferença.

In [19]:
# 3.4 Consistência entre texto e milliseconds (Q14)
diferenca_voltas = (voltas["time"].apply(tempo_para_segundos) * 1000 - voltas["milliseconds"]).abs()
diferenca_paradas = (pit_stops["duration"].apply(tempo_para_segundos) * 1000 - pit_stops["milliseconds"]).abs()
vencedores = resultados[resultados["positionOrder"] == 1].dropna(subset=["time", "milliseconds"])
diferenca_vencedores = (vencedores["time"].apply(tempo_para_segundos) * 1000 - vencedores["milliseconds"]).abs()
inconsistentes_q14 = (diferenca_voltas > 1).sum() + (diferenca_paradas > 1).sum() + (diferenca_vencedores > 1).sum()
print("Voltas com diferença > 1 ms:", (diferenca_voltas > 1).sum(), "de", len(voltas))
print("Paradas com diferença > 1 ms:", (diferenca_paradas > 1).sum(), "de", len(pit_stops))
print("Vencedores com diferença > 1 ms:", (diferenca_vencedores > 1).sum(), "de", len(vencedores))
print("Anos das paradas inconsistentes:", pit_stops.loc[diferenca_paradas > 1, "raceId"].map(ano_da_corrida).value_counts().to_dict())

Voltas com diferença > 1 ms: 0 de 632937
Paradas com diferença > 1 ms: 70 de 12783
Vencedores com diferença > 1 ms: 0 de 1167
Anos das paradas inconsistentes: {2026: 70}


**Resultado:**
- Q3: 8 colunas de data e tempo foram lidas como texto. Sem converter, não dá para comparar tempos nem medir ritmo, o que afeta P2 e P5.
- Q4: os formatos são bem misturados. A coluna de tempo dos resultados tem 21 formatos, porque o vencedor tem o tempo total de prova e os outros pilotos têm só a diferença para ele. A duração das paradas tem 3 formatos.
- Q14: nas voltas e nos vencedores, o texto e os milissegundos batem em todos os casos. Nas paradas, 70 de 12.783 não batem, e todas são de 2026, fora do período escolhido.

### 3.5 Valores impossíveis: Q5 e Q6

Procuramos valores que não podem existir, como posições ou pontos negativos e voltas ou tempos iguais a zero.

In [20]:
# 3.5 Valores impossíveis (Q5)
invalidos = {
    "results: grid < 0": (resultados["grid"] < 0).sum(),
    "results: positionOrder <= 0": (resultados["positionOrder"] <= 0).sum(),
    "results: points < 0": (resultados["points"] < 0).sum(),
    "results: laps < 0": (resultados["laps"] < 0).sum(),
    "results: milliseconds <= 0": (resultados["milliseconds"] <= 0).sum(),
    "qualifying: position <= 0": (classificacao["position"] <= 0).sum(),
    "pit_stops: lap <= 0": (pit_stops["lap"] <= 0).sum(),
    "pit_stops: milliseconds <= 0": (pit_stops["milliseconds"] <= 0).sum(),
    "lap_times: lap <= 0": (voltas["lap"] <= 0).sum(),
    "lap_times: milliseconds <= 0": (voltas["milliseconds"] <= 0).sum(),
    "pit_stops 2014–2025: milliseconds <= 0": (pit_stops.loc[pit_stops["raceId"].isin(ids_2014_2025), "milliseconds"] <= 0).sum(),
}
paradas_zero = pit_stops[pit_stops["milliseconds"] <= 0]
print("Anos das paradas com milliseconds <= 0:", paradas_zero["raceId"].map(ano_da_corrida).value_counts().to_dict())
pd.Series(invalidos, name="registros")

Anos das paradas com milliseconds <= 0: {2026: 70, 2025: 1}


,registros
results: grid < 0,0
results: positionOrder <= 0,0
results: points < 0,0
results: laps < 0,0
results: milliseconds <= 0,0
qualifying: position <= 0,0
pit_stops: lap <= 0,0
pit_stops: milliseconds <= 0,71
lap_times: lap <= 0,0
lap_times: milliseconds <= 0,0


A coluna grid guarda a posição de largada, e nela aparece o valor 0. Parece erro, mas na Fórmula 1 isso indica que o piloto largou da faixa dos boxes, chamada pit lane, e não do grid. Isso acontece por punição ou por troca de peças do carro. Contamos esses casos e vemos como esses pilotos terminaram.

In [21]:
# 3.5 Largadas com grid = 0 (Q6)
grid_zero = resultados_2014_2025[resultados_2014_2025["grid"] == 0].merge(status, on="statusId", how="left")
print("grid = 0 nos dados brutos:", (resultados["grid"] == 0).sum())
print("grid = 0 em 2014–2025:", len(grid_zero))
print("grid = 0 com 0 voltas completadas:", (grid_zero["laps"] == 0).sum())
print("grid = 0 com positionText = W (não largou):", (grid_zero["positionText"] == "W").sum())
grid_zero["status"].value_counts().head(8)

grid = 0 nos dados brutos: 1638
grid = 0 em 2014–2025: 79
grid = 0 com 0 voltas completadas: 7
grid = 0 com positionText = W (não largou): 6


,count
status,
+1 Lap,27
Finished,22
+2 Laps,7
Withdrew,3
Collision damage,3
Brakes,2
Power Unit,2
Accident,2


**Resultado:**
- Q5: não há posição, ponto ou volta negativa. O único valor impossível é a duração zero em 71 paradas. Dessas, 70 são de 2026 e só 1 está no nosso período, no GP da Áustria de 2025. Uma parada não pode durar zero segundos.
- Q6: 79 resultados do período têm grid igual a 0. A maioria terminou a corrida normalmente, o que confirma que são largadas dos boxes e não erros. Mas 6 desses pilotos têm o código W, ou seja, desistiram e nem largaram.

### 3.6 Categorias inconsistentes: Q7 e Q8

A coluna positionText guarda a posição final, mas mistura números com letras. Cada letra é um código: R quer dizer que o piloto abandonou, D que foi desclassificado, W que desistiu antes da largada, N que não foi classificado, E que foi excluído e F que não conseguiu tempo suficiente para largar.

In [22]:
# 3.6 Códigos de positionText (Q7)
codigos_brutos = resultados.loc[~resultados["positionText"].astype(str).str.isdigit(), "positionText"]
codigos_recorte = resultados_2014_2025.loc[~resultados_2014_2025["positionText"].astype(str).str.isdigit(), "positionText"]
pd.DataFrame({"brutos": codigos_brutos.value_counts(), "2014_2025": codigos_recorte.value_counts()}).fillna(0).astype(int)

,brutos,2014_2025
positionText,,
D,157,17
E,9,1
F,1368,0
N,190,1
R,9002,726
W,346,31


A coluna status explica como cada piloto terminou a corrida: se cruzou a linha de chegada, se bateu, se o motor quebrou e assim por diante. Contamos quantas categorias existem e mostramos as 20 mais comuns no período de 2014 a 2025.

In [23]:
# 3.6 Categorias de status (Q8)
status_2014_2025 = resultados_2014_2025.merge(status, on="statusId", how="left")
print("Status distintos na tabela status:", status["status"].nunique())
print("Status distintos usados em 2014–2025:", status_2014_2025["status"].nunique())
status_2014_2025["status"].value_counts().head(20)

Status distintos na tabela status: 141
Status distintos usados em 2014–2025: 68


,count
status,
Finished,2733
+1 Lap,1178
+2 Laps,208
Collision,150
Retired,101
Lapped,89
Accident,85
Engine,81
Collision damage,59


**Resultado:**
- Q7: no período, 776 resultados têm uma letra no lugar da posição: 726 abandonos, 31 desistências, 17 desclassificações, 1 não classificado e 1 excluído. São as mesmas 776 linhas sem posição vistas na Q1.
- Q8: existem 141 categorias de status, e 68 aparecem no período. Muitas dizem quase a mesma coisa com detalhes diferentes, como Engine, Power Unit e Turbo, que são todas falhas ligadas ao motor. Sem agrupar, não dá para calcular a taxa de quebras da P4. O status Retired, com 101 casos, só diz que o piloto abandonou, sem dizer o motivo.

### 3.7 Ligação entre tabelas: Q9, Q10 e Q11

Verificamos se todo código usado em uma tabela existe na tabela de origem. Por exemplo, todo piloto citado nos resultados precisa existir na tabela de pilotos. Um código sem correspondência é chamado de órfão.

In [24]:
# 3.7 Integridade referencial (Q9)
RELACOES = [("results", "raceId", "races"), ("results", "driverId", "drivers"), ("results", "constructorId", "constructors"),
            ("results", "statusId", "status"), ("races", "circuitId", "circuits"), ("qualifying", "raceId", "races"),
            ("qualifying", "driverId", "drivers"), ("qualifying", "constructorId", "constructors"),
            ("pit_stops", "raceId", "races"), ("pit_stops", "driverId", "drivers"),
            ("lap_times", "raceId", "races"), ("lap_times", "driverId", "drivers")]
linhas_orfaos = []
for filha, coluna, mae in RELACOES:
    quantidade = (~tabelas[filha][coluna].isin(tabelas[mae][coluna])).sum()
    linhas_orfaos.append({"tabela": filha, "coluna": coluna, "referencia": mae, "orfaos": quantidade})
orfaos = pd.DataFrame(linhas_orfaos)
orfaos

,tabela,coluna,referencia,orfaos
0,results,raceId,races,0
1,results,driverId,drivers,0
2,results,constructorId,constructors,0
3,results,statusId,status,0
4,races,circuitId,circuits,0
5,qualifying,raceId,races,0
6,qualifying,driverId,drivers,0
7,qualifying,constructorId,constructors,0
8,pit_stops,raceId,races,0
9,pit_stops,driverId,drivers,0


Também conferimos se cada piloto que aparece na classificação, nas paradas ou nas voltas de uma corrida tem um resultado nessa mesma corrida.

In [25]:
# 3.7 Pares corrida + piloto sem resultado
pares_resultados = resultados[["raceId", "driverId"]].drop_duplicates()
pares_sem_resultado = {}
for nome in ["qualifying", "pit_stops", "lap_times"]:
    pares = tabelas[nome][["raceId", "driverId"]].drop_duplicates()
    cruzamento = pares.merge(pares_resultados, how="left", indicator=True)
    pares_sem_resultado[nome] = (cruzamento["_merge"] == "left_only").sum()
pd.Series(pares_sem_resultado, name="pares_sem_resultado")

,pares_sem_resultado
qualifying,1
pit_stops,0
lap_times,0


Q10: contamos quantas corridas do período não têm nenhum registro em cada tabela.

In [26]:
# 3.7 Cobertura das corridas de 2014–2025 (Q10)
cobertura = {}
for nome in ["results", "qualifying", "pit_stops", "lap_times"]:
    cobertura[nome] = (~ids_2014_2025.isin(tabelas[nome]["raceId"])).sum()
print("Corridas no recorte:", len(ids_2014_2025))
display(pd.Series(cobertura, name="corridas_sem_dados"))
sem_pit = corridas[corridas["raceId"].isin(ids_2014_2025) & ~corridas["raceId"].isin(pit_stops["raceId"])]
sem_pit[["raceId", "year", "name", "date"]]

Corridas no recorte: 252


,corridas_sem_dados
results,0
qualifying,0
pit_stops,1
lap_times,0


,raceId,year,name,date
1048,1063,2021,Belgian Grand Prix,2021-08-29


Q11: procuramos corridas cadastradas sem nenhum resultado e contamos quantas corridas tem cada uma das últimas temporadas.

In [27]:
# 3.7 Corridas sem resultado e temporadas recentes (Q11)
sem_resultado = corridas[~corridas["raceId"].isin(resultados["raceId"])]
print("Corridas sem nenhum resultado:", len(sem_resultado))
corridas["year"].value_counts().sort_index().tail(4)

Corridas sem nenhum resultado: 0


,count
year,
2023,22
2024,24
2025,24
2026,15


**Resultado:**
- Q9: não há nenhum código órfão nas 12 ligações conferidas. Só um piloto aparece na classificação sem ter resultado: Lance Stroll no GP da Espanha de 2025, que participou da classificação mas não largou. O registro é real.
- Q10: das 252 corridas do período, só uma não tem paradas: o GP da Bélgica de 2021. Choveu tanto que os carros só andaram atrás do carro de segurança, e ninguém parou para trocar pneus.
- Q11: não há corrida sem resultado. A temporada de 2026 tem só 15 corridas porque ainda não terminou, e por isso fica de fora.

### 3.8 Valores extremos: Q12 e Q13

Q12: o pit stop é a parada nos boxes para trocar pneus. Contando a entrada e a saída da faixa dos boxes, uma parada normal leva de 20 a 30 segundos. Contamos as paradas acima de 60 segundos e vemos em quais corridas elas aparecem.

In [28]:
# 3.8 Duração dos pit stops (Q12)
pit_2014_2025 = pit_stops[pit_stops["raceId"].isin(ids_2014_2025)]
paradas_longas = pit_2014_2025[pit_2014_2025["milliseconds"] > 60000]
print("Paradas no recorte:", len(pit_2014_2025))
print("Paradas acima de 60 s:", len(paradas_longas))
pit_2014_2025["milliseconds"].describe().round(0)

Paradas no recorte: 9181
Paradas acima de 60 s: 509


,milliseconds
count,9181.0
mean,99786.0
std,343716.0
min,0.0
25%,22200.0
50%,23925.0
75%,28021.0
max,3069017.0


In [29]:
# 3.8 Corridas com mais paradas acima de 60 s
corridas_longas = paradas_longas.merge(corridas[["raceId", "year", "name"]], on="raceId")
corridas_longas[["year", "name"]].value_counts().head(8)

year  name                     
2023  Australian Grand Prix        49
2016  Brazilian Grand Prix         36
2021  Saudi Arabian Grand Prix     35
2020  Tuscan Grand Prix            25
2021  Emilia Romagna Grand Prix    22
2014  Japanese Grand Prix          22
2021  British Grand Prix           19
      Azerbaijan Grand Prix        19
Name: count, dtype: int64

Q13: algumas voltas são bem mais lentas que o normal. É o caso da primeira volta, que começa com o carro parado, das voltas com parada nos boxes e das voltas atrás do carro de segurança, que entra na pista depois de um acidente e obriga todos a andar devagar. Para achar essas voltas usamos a regra dos 107% da Fórmula 1: uma volta é considerada lenta quando passa de 107% da mediana das voltas da corrida. A mediana é o valor do meio quando colocamos todos os tempos em ordem.

In [30]:
# 3.8 Voltas muito lentas (Q13)
voltas_2014_2025 = voltas[voltas["raceId"].isin(ids_2014_2025)]
mediana_por_corrida = voltas_2014_2025.groupby("raceId")["milliseconds"].transform("median")
acima_107 = voltas_2014_2025["milliseconds"] > 1.07 * mediana_por_corrida
print("Voltas no recorte:", len(voltas_2014_2025))
print("Voltas acima de 107% da mediana da corrida:", acima_107.sum())
print("Voltas de número 1:", (voltas_2014_2025["lap"] == 1).sum())
voltas_2014_2025["milliseconds"].describe().round(0)

Voltas no recorte: 274801
Voltas acima de 107% da mediana da corrida: 32478
Voltas de número 1: 4952


,milliseconds
count,274801.0
mean,96709.0
std,75500.0
min,55404.0
25%,81824.0
50%,91942.0
75%,102313.0
max,3803459.0


**Resultado:**
- Q12: a parada típica dura 23,9 segundos, mas 509 das 9.181 paradas do período, ou 5,5%, passam de 60 segundos. A maior chega a 3.069 segundos, cerca de 51 minutos. Elas se concentram em corridas interrompidas por bandeira vermelha, quando os carros esperam parados nos boxes. A Austrália em 2023 teve 49 dessas paradas, o Brasil em 2016 teve 36 e a Arábia Saudita em 2021 teve 35. São eventos reais.
- Q13: das 274.801 voltas do período, 32.478, ou 11,8%, passam de 107% da mediana, e há 4.952 primeiras voltas. A volta mais longa tem 3.803 segundos, numa corrida interrompida. Essas voltas aconteceram de verdade, mas atrapalham qualquer medida de ritmo da P5.

### 3.9 Resumo do diagnóstico

Juntamos as 14 verificações em uma tabela. Os números vêm das células anteriores.

In [31]:
# 3.9 Tabela-resumo do diagnóstico (parte 1)
linhas_diagnostico = [
    ["Q1", "Valores ausentes", f"{resultados_2014_2025['position'].isna().sum()} resultados sem posição e {classificacao_2014_2025['q3'].isna().sum()} sem Q3 no recorte", "média"],
    ["Q2", "Duplicatas pela chave", f"{duplicatas['duplicatas_brutos'].sum()} nos brutos; {duplicatas['duplicatas_2014_2025'].sum()} no recorte", "baixa"],
    ["Q3", "Tipos incorretos", f"{(tipos_lidos['tipo_lido'] == 'object').sum()} colunas de data/tempo lidas como texto", "alta"],
    ["Q4", "Formatos de tempo misturados", "m:ss.sss, h:mm:ss.sss, +ss.sss e mm:ss.sss na mesma coluna", "alta"],
    ["Q5", "Valores impossíveis", f"{len(paradas_zero)} paradas com milliseconds = 0 nos brutos; {invalidos['pit_stops 2014–2025: milliseconds <= 0']} no recorte", "média"],
    ["Q6", "grid = 0", f"{len(grid_zero)} resultados no recorte (largada do pit lane)", "média"],
    ["Q7", "Códigos em positionText", f"{codigos_recorte.nunique()} códigos de letra no recorte ({len(codigos_recorte)} linhas)", "média"],
]
len(linhas_diagnostico)

7

In [32]:
# 3.9 Tabela-resumo do diagnóstico (parte 2)
linhas_diagnostico = linhas_diagnostico + [
    ["Q8", "Status fragmentado", f"{status_2014_2025['status'].nunique()} categorias usadas no recorte", "alta"],
    ["Q9", "Integridade referencial", f"{orfaos['orfaos'].sum()} ids órfãos; {sum(pares_sem_resultado.values())} pares sem resultado", "baixa"],
    ["Q10", "Cobertura do recorte", f"{cobertura['pit_stops']} corrida sem pit stops; {cobertura['lap_times']} sem voltas", "baixa"],
    ["Q11", "Corridas futuras/incompletas", f"{len(sem_resultado)} sem resultado; temporada 2026 incompleta", "média"],
    ["Q12", "Pit stops muito longos", f"{len(paradas_longas)} paradas > 60 s no recorte", "média"],
    ["Q13", "Voltas atípicas", f"{acima_107.sum()} voltas > 107% da mediana no recorte", "alta"],
    ["Q14", "Texto x milliseconds", f"{inconsistentes_q14} inconsistências, todas em paradas de 2026", "baixa"],
]
diagnostico = pd.DataFrame(linhas_diagnostico, columns=["ID", "Verificação", "Resultado", "Prioridade"])
diagnostico

,ID,Verificação,Resultado,Prioridade
0,Q1,Valores ausentes,776 resultados sem posição e 2650 sem Q3 no re...,média
1,Q2,Duplicatas pela chave,91 nos brutos; 0 no recorte,baixa
2,Q3,Tipos incorretos,8 colunas de data/tempo lidas como texto,alta
3,Q4,Formatos de tempo misturados,"m:ss.sss, h:mm:ss.sss, +ss.sss e mm:ss.sss na ...",alta
4,Q5,Valores impossíveis,71 paradas com milliseconds = 0 nos brutos; 1 ...,média
5,Q6,grid = 0,79 resultados no recorte (largada do pit lane),média
6,Q7,Códigos em positionText,5 códigos de letra no recorte (776 linhas),média
7,Q8,Status fragmentado,68 categorias usadas no recorte,alta
8,Q9,Integridade referencial,0 ids órfãos; 1 pares sem resultado,baixa
9,Q10,Cobertura do recorte,1 corrida sem pit stops; 0 sem voltas,baixa


**Resposta do grupo:**

A base tem 9 tabelas, e a principal é a de resultados, com uma linha para cada piloto em cada corrida. No período de 2014 a 2025 há 252 corridas, e todas aparecem nas tabelas principais. Quase não há erros de verdade, mas vários pontos atrapalham as perguntas se não forem tratados:
- Os tempos estão guardados como texto e em formatos misturados. Isso impede comparar os tempos de classificação da P2 e medir o ritmo da P5.
- Os status estão espalhados em 68 categorias, o que impede calcular a taxa de quebras por equipe da P4.
- As voltas muito lentas e as paradas durante bandeira vermelha distorcem o ritmo da P5 e a análise de estratégia da P3.
- O grid igual a 0 precisa ser entendido como largada dos boxes para comparar largada e chegada na P1 e na P3.
- A temporada de 2026 está incompleta e distorceria a comparação entre os períodos.
- Repetições, códigos órfãos e valores impossíveis quase não aparecem no período escolhido.

## 4. Limpeza e preparação dos dados

Realize a limpeza necessária ao projeto. Verifique valores ausentes, duplicados e inconsistências e trate os problemas encontrados, quando necessário.

Cada passo abaixo é uma decisão registrada, de D1 a D13, ligada a um problema encontrado no diagnóstico. Depois de cada passo há uma conferência mostrando que a mudança funcionou.

### 4.1 Recorte do período: D1

Ficamos só com as corridas de 2014 a 2025 e filtramos as outras tabelas para manter apenas essas corridas. Isso cobre a era híbrida inteira e deixa de fora a temporada de 2026, que ainda não terminou.

In [33]:
# 4.1 Recorte temporal
corridas_limpo = corridas[corridas["year"].between(2014, 2025)].copy()
ids_recorte = corridas_limpo["raceId"]
resultados_limpo = resultados[resultados["raceId"].isin(ids_recorte)].copy()
classificacao_limpo = classificacao[classificacao["raceId"].isin(ids_recorte)].copy()
pit_stops_limpo = pit_stops[pit_stops["raceId"].isin(ids_recorte)].copy()
voltas_limpo = voltas[voltas["raceId"].isin(ids_recorte)].copy()
print("Corridas:", len(corridas), "->", len(corridas_limpo))
print("Resultados:", len(resultados), "->", len(resultados_limpo))
print("Classificação:", len(classificacao), "->", len(classificacao_limpo))
print("Pit stops:", len(pit_stops), "->", len(pit_stops_limpo))
print("Voltas:", len(voltas), "->", len(voltas_limpo))

Corridas: 1164 -> 252
Resultados: 27568 -> 5105
Classificação: 11298 -> 5089
Pit stops: 12783 -> 9181
Voltas: 632937 -> 274801


In [34]:
# 4.1 Verificação do recorte
assert corridas_limpo["year"].between(2014, 2025).all()
assert resultados_limpo["raceId"].isin(ids_recorte).all()
corridas_limpo["year"].agg(["min", "max", "nunique"])

,year
min,2014
max,2025
nunique,12


### 4.2 Tipos de dados e repetições: D2 e D11

Transformamos a data da corrida de texto em data de verdade. As colunas de posição e de número do carro foram lidas como números decimais só porque têm vazios, então passamos para um tipo inteiro que aceita vazio.

In [35]:
# 4.2 Correção de tipos
corridas_limpo["date"] = pd.to_datetime(corridas_limpo["date"], format="%Y-%m-%d")
resultados_limpo["position"] = resultados_limpo["position"].astype("Int64")
resultados_limpo["number"] = resultados_limpo["number"].astype("Int64")
print(corridas_limpo["date"].dtype, resultados_limpo["position"].dtype, resultados_limpo["number"].dtype)
corridas_limpo[["raceId", "year", "round", "date"]].head(3)

datetime64[ns] Int64 Int64


,raceId,year,round,date
897,900,2014,1,2014-03-16
898,901,2014,2,2014-03-30
899,902,2014,3,2014-04-06


Repetimos, só no período escolhido, a busca por registros repetidos e códigos órfãos.

In [36]:
# 4.2 Duplicatas e órfãos no recorte
verificacao_d11 = {
    "duplicatas em resultados": resultados_limpo.duplicated(subset=["raceId", "driverId"]).sum(),
    "duplicatas em classificação": classificacao_limpo.duplicated(subset=["raceId", "driverId"]).sum(),
    "duplicatas em pit stops": pit_stops_limpo.duplicated(subset=["raceId", "driverId", "stop"]).sum(),
    "duplicatas em voltas": voltas_limpo.duplicated(subset=["raceId", "driverId", "lap"]).sum(),
    "pilotos órfãos": (~resultados_limpo["driverId"].isin(pilotos["driverId"])).sum(),
    "equipes órfãs": (~resultados_limpo["constructorId"].isin(equipes["constructorId"])).sum(),
    "status órfãos": (~resultados_limpo["statusId"].isin(status["statusId"])).sum(),
}
pd.Series(verificacao_d11, name="registros")

,registros
duplicatas em resultados,0
duplicatas em classificação,0
duplicatas em pit stops,0
duplicatas em voltas,0
pilotos órfãos,0
equipes órfãs,0
status órfãos,0


**Resultado:** a data agora é do tipo data e as posições são números inteiros. No período não há repetições nem códigos órfãos, então nenhuma linha foi removida. Mantivemos o registro de classificação de Lance Stroll na Espanha em 2025, mesmo sem resultado, porque ele é verdadeiro.

### 4.3 Tempos em segundos e vazios mantidos: D3, D13 e D4

Usamos a função tempo_para_segundos nos tempos das três fases da classificação e criamos colunas novas em segundos, chamadas q1_s, q2_s e q3_s.

In [37]:
# 4.3 Tempos da classificação em segundos
for coluna in ["q1", "q2", "q3"]:
    classificacao_limpo[f"{coluna}_s"] = classificacao_limpo[coluna].apply(tempo_para_segundos)
classificacao_limpo[["q1", "q1_s", "q2", "q2_s", "q3", "q3_s"]].head()

,q1,q1_s,q2,q2_s,q3,q3_s
5857,1:31.699,91.699,1:42.890,102.890,1:44.231,104.231
5858,1:30.775,90.775,1:42.295,102.295,1:44.548,104.548
5859,1:32.564,92.564,1:42.264,102.264,1:44.595,104.595
5860,1:30.949,90.949,1:43.247,103.247,1:45.745,105.745
5861,1:31.388,91.388,1:42.805,102.805,1:45.819,105.819


Nas voltas e nas paradas já existe uma coluna numérica em milissegundos. Basta dividir por 1.000 para ter o tempo em segundos.

In [38]:
# 4.3 Voltas e paradas em segundos
voltas_limpo["segundos"] = voltas_limpo["milliseconds"] / 1000
pit_stops_limpo["segundos"] = pit_stops_limpo["milliseconds"] / 1000
print(voltas_limpo["segundos"].dtype, pit_stops_limpo["segundos"].dtype)
voltas_limpo[["raceId", "driverId", "lap", "time", "segundos"]].head(3)

float64 float64


,raceId,driverId,lap,time,segundos
72130,900,3,1,1:42.038,102.038
72131,900,3,2,1:37.687,97.687
72132,900,3,3,1:35.765,95.765


D13: no período existe uma parada com duração zero e sem duração em texto. A parada aconteceu e conta no número de paradas da P3, mas não sabemos quanto tempo ela durou. Por isso mantemos a linha e deixamos a duração vazia, em vez de registrar um zero que não é verdade.

In [39]:
# 4.3 Parada com duração zero (D13)
duracao_zero = pit_stops_limpo["milliseconds"] <= 0
print("Paradas com milliseconds <= 0 no recorte:", duracao_zero.sum())
pit_stops_limpo.loc[duracao_zero, "segundos"] = np.nan
display(pit_stops_limpo[duracao_zero])
print("Paradas com segundos vazio:", pit_stops_limpo["segundos"].isna().sum())

Paradas com milliseconds <= 0 no recorte: 1


,raceId,driverId,stop,lap,time,duration,milliseconds,segundos
11807,1155,852,2,30,15:55:58,NaN,0,NaN


Paradas com segundos vazio: 1


D4: um tempo vazio no Q2 ou no Q3 mostra que o piloto foi eliminado antes, e uma posição vazia mostra que ele não terminou a corrida. Esses vazios têm significado, por isso não preenchemos nada. Conferimos que a conversão não criou vazios novos.

In [40]:
# 4.3 Verificação dos nulos mantidos
for coluna in ["q1", "q2", "q3"]:
    nulos_texto = classificacao_limpo[coluna].isna().sum()
    nulos_segundos = classificacao_limpo[f"{coluna}_s"].isna().sum()
    assert nulos_texto == nulos_segundos
    print(f"{coluna}: vazios no texto = {nulos_texto} | vazios em segundos = {nulos_segundos}")
print("position vazia em resultados:", resultados_limpo["position"].isna().sum())

q1: vazios no texto = 78 | vazios em segundos = 78
q2: vazios no texto = 1364 | vazios em segundos = 1364
q3: vazios no texto = 2650 | vazios em segundos = 2650
position vazia em resultados: 776


**Resultado:** os tempos da classificação agora estão em segundos. Por exemplo, 1:31.699 virou 91,699. A conversão não criou nenhum vazio novo: continuam 78 vazios no Q1, 1.364 no Q2 e 2.650 no Q3. As 776 posições vazias dos resultados foram mantidas, e a parada com duração zero ficou com a duração vazia.

### 4.4 Agrupamento dos status: D5

Primeiro trazemos o texto do status para a tabela de resultados. A opção validate garante que cada resultado encontre no máximo um status, sem duplicar linhas.

In [41]:
# 4.4 Texto do status nos resultados
linhas_antes = len(resultados_limpo)
resultados_limpo = resultados_limpo.merge(status, on="statusId", how="left", validate="many_to_one")
assert len(resultados_limpo) == linhas_antes
resultados_limpo[["raceId", "driverId", "statusId", "status"]].head(3)

,raceId,driverId,statusId,status
0,900,3,1,Finished
1,900,825,1,Finished
2,900,18,1,Finished


Agrupamos os status em 5 categorias: Terminou, Retardatário, Acidente, Mecânico e Outro. Retardatário é o piloto que terminou a corrida, mas com uma ou mais voltas de atraso em relação ao vencedor. Tudo o que não se encaixa em Terminou, Retardatário, Acidente ou Outro é tratado como quebra do carro, na categoria Mecânico. Mantemos a coluna de status original para que qualquer pessoa possa conferir o agrupamento.

In [42]:
# 4.4 Regra de agrupamento
ACIDENTE = ["Accident", "Collision", "Collision damage", "Spun off", "Damage", "Debris", "Puncture"]
OUTRO = ["Disqualified", "Did not qualify", "Did not start", "Withdrew", "Not classified",
         "Excluded", "Retired", "Illness", "Driver unwell", "107% Rule"]

def agrupar_status(texto):
    if texto == "Finished":
        return "Terminou"
    if texto.startswith("+") or texto == "Lapped":
        return "Retardatário"
    if texto in ACIDENTE:
        return "Acidente"
    if texto in OUTRO:
        return "Outro"
    return "Mecânico"

resultados_limpo["status_grupo"] = resultados_limpo["status"].apply(agrupar_status)
resultados_limpo["status_grupo"].value_counts()

,count
status_grupo,
Terminou,2733
Retardatário,1511
Mecânico,407
Acidente,315
Outro,139


Mostramos cada status original ao lado do grupo em que ele ficou, para revisar à mão se algum caiu no lugar errado.

In [43]:
# 4.4 Revisão status x grupo
revisao_status = resultados_limpo[["status_grupo", "status"]].value_counts().sort_index()
revisao_status.to_frame("registros")

registros
status_grupo status                     
Acidente     Accident                 85
             Collision               150
             Collision damage         59
             Damage                    2
             Debris                    1
             Puncture                  7
             Spun off                 11
Mecânico     Battery                   5
             Brake duct                1
             Brakes                   49
             Clutch                    2
             Cooling system            1
             Differential              1
             Driveshaft                2
             Drivetrain                1
             ERS                       5
             Electrical               14
             Electronics               4
             Engine                   81
             Exhaust                   5
             Front wing                3
             Fuel leak                 2
             Fuel pressure             5
             Fuel pump                 1
             Fuel system               1
             Gearbox                  42
             Hydraulics               15
             Mechanical                7
             Oil leak                 10
             Oil pressure              2
             Out of fuel               1
             Overheating               8
             Power Unit               42
             Power loss               13
             Radiator                  3
             Rear wing                 3
             Seat                      1
             Spark plugs               1
             Steering                  2
             Suspension               25
             Technical                 2
             Throttle                  2
             Transmission              3
             Turbo                     6
             Tyre                      4
             Undertray                 4
             Vibrations                2
             Water leak                4
             Water pressure            7
             Water pump                1
             Wheel                    11
             Wheel nut                 3
Outro        Did not start             3
             Disqualified             17
             Excluded                  1
             Illness                   3
             Retired                 101
             Withdrew                 14
Retardatário +1 Lap                 1178
             +2 Laps                 208
             +3 Laps                  24
             +4 Laps                   6
             +5 Laps                   1
             +6 Laps                   2
             +7 Laps                   1
             +8 Laps                   2
             Lapped                   89
Terminou     Finished               2733

**Resultado da revisão:** os 68 status do período ficaram assim: Terminou com 2.733 resultados, Retardatário com 1.511, Mecânico com 407, Acidente com 315 e Outro com 139. Alguns casos são duvidosos e ficam registrados como limitação. As quebras de asa, com 6 casos, podem ser defeito ou consequência de um toque em outro carro. Tyre, que é falha de pneu, ficou em Mecânico, enquanto Puncture, que é pneu furado, ficou em Acidente. Retired, com 101 casos, não diz o motivo do abandono e ficou em Outro.

### 4.5 Marcação dos valores extremos: D6, D7 e D8

Em vez de apagar os valores extremos, criamos colunas de verdadeiro ou falso que marcam cada caso. As linhas continuam na base, e quem for analisar decide se usa ou não.

D6: a coluna largou_box marca quem largou dos boxes, ou seja, quem tem grid igual a 0. Ficam de fora os 6 pilotos com código W, que desistiram antes da largada.

In [44]:
# 4.5 Largada do pit lane (D6)
resultados_limpo["largou_box"] = (resultados_limpo["grid"] == 0) & (resultados_limpo["positionText"] != "W")
resultados_limpo["largou_box"].value_counts()

,count
largou_box,
False,5032
True,73


D7: a coluna parada_longa marca as paradas acima de 60 segundos, que em geral acontecem quando a corrida é interrompida.

In [45]:
# 4.5 Paradas longas (D7)
pit_stops_limpo["parada_longa"] = pit_stops_limpo["milliseconds"] > 60000
pit_stops_limpo["parada_longa"].value_counts()

,count
parada_longa,
False,8672
True,509


D8: a coluna volta_atipica marca três situações: a primeira volta, as voltas ligadas a uma parada e as voltas acima de 107% da mediana da corrida. Cada parada afeta duas voltas: a volta em que o piloto entra nos boxes e a seguinte, em que ele sai. Primeiro montamos a lista dessas voltas.

In [46]:
# 4.5 Voltas de entrada e saída do pit
voltas_entrada = pit_stops_limpo[["raceId", "driverId", "lap"]]
voltas_saida = voltas_entrada.assign(lap=voltas_entrada["lap"] + 1)
voltas_com_pit = pd.concat([voltas_entrada, voltas_saida]).drop_duplicates()
voltas_limpo = voltas_limpo.merge(voltas_com_pit, on=["raceId", "driverId", "lap"], how="left", indicator="origem")
voltas_limpo["volta_pit"] = voltas_limpo["origem"] == "both"
voltas_limpo = voltas_limpo.drop(columns="origem")
voltas_limpo["volta_pit"].value_counts()

,count
volta_pit,
False,256936
True,17865


In [47]:
# 4.5 Voltas atípicas (D8)
mediana_corrida = voltas_limpo.groupby("raceId")["milliseconds"].transform("median")
voltas_limpo["volta_lenta"] = voltas_limpo["milliseconds"] > 1.07 * mediana_corrida
voltas_limpo["volta_atipica"] = (voltas_limpo["lap"] == 1) | voltas_limpo["volta_pit"] | voltas_limpo["volta_lenta"]
percentual_atipicas = voltas_limpo["volta_atipica"].mean() * 100
print("Voltas marcadas como atípicas:", voltas_limpo["volta_atipica"].sum(), "de", len(voltas_limpo), f"({percentual_atipicas:.1f}%)")
voltas_limpo[["volta_pit", "volta_lenta", "volta_atipica"]].sum()

Voltas marcadas como atípicas: 38998 de 274801 (14.2%)


,0
volta_pit,17865
volta_lenta,32478
volta_atipica,38998


**Resultado:**
- D6: 73 resultados foram marcados como largada dos boxes, que são os 79 com grid igual a 0 menos os 6 que não largaram.
- D7: 509 paradas foram marcadas como longas, 5,5% do total. Nenhuma foi apagada.
- D8: 38.998 das 274.801 voltas, ou 14,2%, foram marcadas como atípicas. Uma mesma volta pode ter mais de um motivo. As voltas que sobram mostram o ritmo normal dos pilotos, que será usado na P5.

**Resposta do grupo:**

- **Período:** ficamos só com as temporadas de 2014 a 2025. Os resultados passaram de 27.568 para 5.105 linhas.
- **Tipos:** a data virou data de verdade, as posições viraram números inteiros e os tempos em texto viraram segundos.
- **Valores vazios:** foram mantidos porque têm significado, como o piloto eliminado na classificação ou o que não terminou a corrida. A única parada com duração zero ficou com a duração vazia.
- **Status:** as 68 categorias do período viraram 5 grupos, e o texto original foi mantido.
- **Valores extremos:** não apagamos nada. Marcamos 73 largadas dos boxes, 509 paradas longas e 38.998 voltas atípicas.
- **Repetições e códigos órfãos:** não existiam no período, então não foi preciso tratar.

## 5. Transformação dos dados

Realize as transformações necessárias à análise, como criação de colunas, filtros, agrupamentos, conversões ou junções, quando aplicáveis.

Algumas transformações já aparecem na seção 4 porque fazem parte da limpeza, como a conversão dos tempos e o agrupamento dos status. Aqui criamos as colunas de apoio, juntamos as tabelas e preparamos os arquivos finais.

### 5.1 Colunas de apoio: D9 e D10

Trazemos o ano da corrida para a tabela de resultados e criamos a coluna periodo, que separa as corridas de antes e de depois da mudança de regras de 2022.

In [48]:
# 5.1 Período do regulamento (D9)
resultados_limpo = resultados_limpo.merge(corridas_limpo[["raceId", "year", "round", "date", "name", "circuitId"]], on="raceId", how="left", validate="many_to_one")
resultados_limpo["periodo"] = np.where(resultados_limpo["year"] >= 2022, "2022-2025", "2014-2021")
resultados_limpo[["year", "periodo"]].drop_duplicates().groupby("periodo")["year"].agg(["min", "max"])

,min,max
periodo,,
2014-2021,2014,2021
2022-2025,2022,2025


Criamos a coluna terminou, que é verdadeira para quem completou a corrida, ou seja, para os grupos Terminou e Retardatário.

In [49]:
# 5.1 Piloto completou a corrida (D10)
resultados_limpo["terminou"] = resultados_limpo["status_grupo"].isin(["Terminou", "Retardatário"])
pd.crosstab(resultados_limpo["status_grupo"], resultados_limpo["terminou"])

terminou,False,True
status_grupo,,
Acidente,315,0
Mecânico,407,0
Outro,139,0
Retardatário,0,1511
Terminou,0,2733


**Resultado:** todas as corridas caíram em um dos dois períodos, sem nenhum ano fora deles. A coluna terminou é verdadeira em 4.244 resultados e falsa nos outros 861.

### 5.2 Junção das tabelas: D12

Juntamos aos resultados os nomes do circuito, do piloto e da equipe, que estavam em outras tabelas. A junção não perde nenhum resultado, e a opção validate garante que nenhuma linha seja duplicada.

In [50]:
# 5.2 Nomes de circuito, piloto e equipe
circuitos_nomes = circuitos[["circuitId", "name", "country"]].rename(columns={"name": "circuito", "country": "pais"})
pilotos_nomes = pilotos[["driverId", "forename", "surname"]].copy()
pilotos_nomes["piloto"] = pilotos_nomes["forename"] + " " + pilotos_nomes["surname"]
equipes_nomes = equipes[["constructorId", "name"]].rename(columns={"name": "equipe"})
linhas_antes = len(resultados_limpo)
resultados_limpo = resultados_limpo.rename(columns={"name": "corrida"})
resultados_limpo = resultados_limpo.merge(circuitos_nomes, on="circuitId", how="left", validate="many_to_one")
resultados_limpo = resultados_limpo.merge(pilotos_nomes[["driverId", "piloto"]], on="driverId", how="left", validate="many_to_one")
resultados_limpo = resultados_limpo.merge(equipes_nomes, on="constructorId", how="left", validate="many_to_one")
assert len(resultados_limpo) == linhas_antes
resultados_limpo[["year", "corrida", "circuito", "piloto", "equipe", "grid", "positionOrder", "status_grupo"]].head()

,year,corrida,circuito,piloto,equipe,grid,positionOrder,status_grupo
0,2014,Australian Grand Prix,Albert Park Grand Prix Circuit,Nico Rosberg,Mercedes,3,1,Terminou
1,2014,Australian Grand Prix,Albert Park Grand Prix Circuit,Kevin Magnussen,McLaren,4,2,Terminou
2,2014,Australian Grand Prix,Albert Park Grand Prix Circuit,Jenson Button,McLaren,10,3,Terminou
3,2014,Australian Grand Prix,Albert Park Grand Prix Circuit,Fernando Alonso,Ferrari,5,4,Terminou
4,2014,Australian Grand Prix,Albert Park Grand Prix Circuit,Valtteri Bottas,Williams,15,5,Terminou


As tabelas de classificação, paradas e voltas também recebem o ano da corrida, para facilitar filtros depois.

In [51]:
# 5.2 Ano nas demais tabelas
anos_corridas = corridas_limpo[["raceId", "year"]]
classificacao_limpo = classificacao_limpo.merge(anos_corridas, on="raceId", how="left", validate="many_to_one")
pit_stops_limpo = pit_stops_limpo.merge(anos_corridas, on="raceId", how="left", validate="many_to_one")
voltas_limpo = voltas_limpo.merge(anos_corridas, on="raceId", how="left", validate="many_to_one")
print(classificacao_limpo["year"].isna().sum(), pit_stops_limpo["year"].isna().sum(), voltas_limpo["year"].isna().sum())

0 0 0


### 5.3 Seleção das colunas finais

Escolhemos as colunas que vão para cada um dos 4 arquivos finais.

| Arquivo | Cada linha representa |
|---|---|
| resultados_limpo.csv | um piloto em uma corrida |
| classificacao_limpo.csv | um piloto na classificação de uma corrida |
| pit_stops_limpo.csv | uma parada nos boxes |
| voltas_limpo.csv | uma volta de um piloto |

In [52]:
# 5.3 Seleção das colunas finais
COLUNAS_RESULTADOS = ["raceId", "year", "round", "date", "periodo", "corrida", "circuito", "pais", "driverId", "piloto",
                      "constructorId", "equipe", "grid", "largou_box", "positionOrder", "position", "points", "laps",
                      "status", "status_grupo", "terminou"]
resultados_final = resultados_limpo[COLUNAS_RESULTADOS]
classificacao_final = classificacao_limpo[["raceId", "year", "driverId", "constructorId", "position", "q1_s", "q2_s", "q3_s"]]
pit_stops_final = pit_stops_limpo[["raceId", "year", "driverId", "stop", "lap", "segundos", "parada_longa"]]
voltas_final = voltas_limpo[["raceId", "year", "driverId", "lap", "position", "segundos", "volta_atipica"]]
for nome, tabela in [("resultados", resultados_final), ("classificacao", classificacao_final), ("pit_stops", pit_stops_final), ("voltas", voltas_final)]:
    print(f"{nome:14} {tabela.shape}")

resultados     (5105, 21)
classificacao  (5089, 8)
pit_stops      (9181, 7)
voltas         (274801, 7)


### 5.4 Validações finais

Usamos assert, um comando que interrompe o notebook com erro se uma condição for falsa. Se todas passarem, aparece uma mensagem de confirmação. Começamos conferindo se não há registros repetidos e se todos os anos estão entre 2014 e 2025.

In [53]:
# 5.4 Chaves únicas e anos
assert not resultados_final.duplicated(subset=["raceId", "driverId"]).any()
assert not classificacao_final.duplicated(subset=["raceId", "driverId"]).any()
assert not pit_stops_final.duplicated(subset=["raceId", "driverId", "stop"]).any()
assert not voltas_final.duplicated(subset=["raceId", "driverId", "lap"]).any()
for tabela in [resultados_final, classificacao_final, pit_stops_final, voltas_final]:
    assert tabela["year"].between(2014, 2025).all()
print("Chaves únicas e anos conferidos")

Chaves únicas e anos conferidos


Depois conferimos os tipos das colunas, os valores permitidos em cada uma, a ligação entre as tabelas e se as colunas principais estão sem vazios.

In [54]:
# 5.4 Tipos, domínios, integridade e chaves sem vazios
assert resultados_final["date"].dtype == "datetime64[ns]"
for coluna in ["q1_s", "q2_s", "q3_s"]:
    assert classificacao_final[coluna].dtype == "float64"
assert pit_stops_final["segundos"].dtype == "float64" and voltas_final["segundos"].dtype == "float64"
assert set(resultados_final["status_grupo"]) == {"Terminou", "Retardatário", "Acidente", "Mecânico", "Outro"}
assert set(resultados_final["periodo"]) == {"2014-2021", "2022-2025"}
assert (resultados_final["grid"] >= 0).all()
assert (pit_stops_final["segundos"].dropna() > 0).all() and (voltas_final["segundos"] > 0).all()
assert pit_stops_final["segundos"].isna().sum() == 1 and voltas_final["segundos"].notna().all()
for tabela in [classificacao_final, pit_stops_final, voltas_final]:
    assert tabela["raceId"].isin(resultados_final["raceId"]).all()
assert resultados_final[["raceId", "driverId", "constructorId", "grid", "positionOrder", "status_grupo"]].notna().all().all()
print("Todas as validações passaram")

Todas as validações passaram


### 5.5 Comparação antes e depois

Comparamos os dados logo após a ingestão, com todos os anos, e depois da preparação, só com o período tratado.

In [55]:
# 5.5 Tabela antes x depois (parte 1)
linhas_comparacao = [
    ["Registros de resultados", len(resultados), len(resultados_final), "recorte 2014–2025"],
    ["Registros de classificação", len(classificacao), len(classificacao_final), "recorte 2014–2025"],
    ["Registros de pit stops", len(pit_stops), len(pit_stops_final), "recorte 2014–2025"],
    ["Registros de voltas", len(voltas), len(voltas_final), "recorte 2014–2025"],
    ["Atributos de resultados", resultados.shape[1], resultados_final.shape[1], "nomes, período, grupos e marcações; removidas colunas não usadas"],
    ["Categorias de status", status["status"].nunique(), resultados_final["status_grupo"].nunique(), f"{status_2014_2025['status'].nunique()} usadas no recorte -> 5 grupos (original mantido)"],
]
len(linhas_comparacao)

6

In [56]:
# 5.5 Tabela antes x depois (parte 2)
colunas_texto_tempo = (tipos_lidos["tipo_lido"] == "object").sum()
paradas_marcadas = pit_stops_final["parada_longa"].sum()
voltas_marcadas = voltas_final["volta_atipica"].sum()
linhas_comparacao = linhas_comparacao + [
    ["Colunas de data/tempo em texto", colunas_texto_tempo, 0, "date e q1–q3 convertidas; voltas e paradas usam milliseconds; time e fastestLapTime de results não seguem"],
    ["Duplicatas pela chave", duplicatas["duplicatas_brutos"].sum(), 0, "nos brutos são de 1950–1978 (carros compartilhados); nenhuma no recorte"],
    ["Paradas com duração 0", len(paradas_zero), 0, "70 eram de 2026 (fora do recorte); 1 virou vazio (D13)"],
    ["Ids órfãos", orfaos["orfaos"].sum(), 0, "nenhuma ação necessária"],
    ["Paradas longas marcadas", 0, paradas_marcadas, f"{paradas_marcadas / len(pit_stops_final) * 100:.1f}% das paradas do recorte"],
    ["Voltas atípicas marcadas", 0, voltas_marcadas, f"{voltas_marcadas / len(voltas_final) * 100:.1f}% das voltas do recorte"],
    ["Largadas do pit lane marcadas", 0, resultados_final["largou_box"].sum(), "grid = 0 mantido e marcado"],
]
antes_depois = pd.DataFrame(linhas_comparacao, columns=["Medida", "Após ingestão", "Após preparação", "Explicação"])
antes_depois

,Medida,Após ingestão,Após preparação,Explicação
0,Registros de resultados,27568,5105,recorte 2014–2025
1,Registros de classificação,11298,5089,recorte 2014–2025
2,Registros de pit stops,12783,9181,recorte 2014–2025
3,Registros de voltas,632937,274801,recorte 2014–2025
4,Atributos de resultados,18,21,"nomes, período, grupos e marcações; removidas ..."
5,Categorias de status,141,5,68 usadas no recorte -> 5 grupos (original man...
6,Colunas de data/tempo em texto,8,0,date e q1–q3 convertidas; voltas e paradas usa...
7,Duplicatas pela chave,91,0,nos brutos são de 1950–1978 (carros compartilh...
8,Paradas com duração 0,71,0,70 eram de 2026 (fora do recorte); 1 virou vaz...
9,Ids órfãos,0,0,nenhuma ação necessária


**Resultado:** os resultados passaram de 27.568 para 5.105 linhas por causa do recorte de período. As 68 categorias de status do período viraram 5 grupos, e nenhuma coluna de tempo ficou em texto nos arquivos finais. Nenhum registro do período foi apagado por problema de qualidade: os casos estranhos foram marcados, e a única parada sem duração ficou vazia, com justificativa.

### 5.6 Exportação

Salvamos as 4 tabelas em arquivos CSV na pasta data/processed. Depois lemos um dos arquivos de novo para confirmar que ele foi gravado direito.

In [57]:
# 5.6 Exportação dos arquivos finais
SAIDAS = {"resultados_limpo.csv": resultados_final, "classificacao_limpo.csv": classificacao_final,
          "pit_stops_limpo.csv": pit_stops_final, "voltas_limpo.csv": voltas_final}
for nome_arquivo, tabela in SAIDAS.items():
    tabela.to_csv(PASTA_PROCESSED / nome_arquivo, index=False, encoding="utf-8")
    print("Salvo:", nome_arquivo, tabela.shape)

relido = pd.read_csv(PASTA_PROCESSED / "resultados_limpo.csv")
print("Relido resultados_limpo.csv:", relido.shape)

Salvo: resultados_limpo.csv (5105, 21)
Salvo: classificacao_limpo.csv (5089, 8)
Salvo: pit_stops_limpo.csv (9181, 7)
Salvo: voltas_limpo.csv (274801, 7)
Relido resultados_limpo.csv: (5105, 21)


**Resposta do grupo:**

- **Colunas novas:** periodo, para comparar antes e depois de 2022, e terminou, para saber quem completou a corrida. Na seção 4 já tinham sido criadas status_grupo, com os 5 grupos de status, e as marcações largou_box, parada_longa e volta_atipica.
- **Junções:** os resultados ganharam o ano, o nome da corrida, do circuito, do piloto e da equipe. As outras tabelas ganharam o ano.
- **Filtros:** ficaram só as temporadas de 2014 a 2025, e cada arquivo final tem só as colunas que serão usadas.
- **Resultado:** 4 arquivos validados. Resultados com 5.105 linhas e 21 colunas, classificação com 5.089 linhas e 8 colunas, paradas com 9.181 linhas e 7 colunas e voltas com 274.801 linhas e 7 colunas.

## 7. Considerações do RA1

Registre brevemente:

- O que o grupo descobriu sobre os dados.
- Quais dificuldades ou limitações encontrou.
- Como os dados preparados poderão ser utilizados nas próximas etapas.

**Resposta do grupo:**

**O que descobrimos sobre os dados:** a base é completa para as temporadas de 2014 a 2025, com as 252 corridas presentes em todas as tabelas principais. Quase não há erros de verdade, como repetições ou códigos órfãos. Os maiores problemas estavam no formato: tempos guardados como texto, categorias de status muito espalhadas e valores extremos que são reais, como paradas durante bandeira vermelha.

**Dificuldades e limitações:**
- Não há dados sobre o tipo de pneu, o clima ou os momentos com carro de segurança. O carro de segurança só aparece de forma indireta, pelas voltas lentas.
- A regra dos 107% é uma aproximação e pode marcar ou deixar passar algumas voltas.
- O agrupamento de status é uma interpretação nossa. O status Retired, por exemplo, não diz o motivo do abandono.
- O conjunto de dados muda a cada corrida, por isso fixamos a versão 9.

**Uso nas próximas etapas:** os 4 arquivos finais estão prontos para responder às perguntas. Os resultados servem para P1, P3 e P4, a classificação para P2, e as paradas e voltas para P3 e P5. As colunas de marcação deixam cada análise escolher se inclui ou não os casos extremos.

---

# RA2 — Streaming Data Processing and Time Windows

Utilize os dados e o contexto do projeto para demonstrar o processamento em streaming e a aplicação de janelas temporais. Escolha um dos cenários:

- **Cenário A — Dados estáticos:** simule a chegada de novos registros utilizando dados artificiais coerentes com o contexto do projeto.
- **Cenário B — Dados em streaming:** utilize sua própria fonte de dados contínuos, sem necessidade de simulação.

## 8. Objetivo da análise em streaming

Responda brevemente:

- O que o grupo deseja monitorar ou analisar?
- Qual informação espera obter com o processamento em streaming?

**Resposta do grupo:**

Queremos acompanhar o ritmo dos pilotos durante a corrida. A cada 5 minutos, calculamos o tempo médio de volta de cada piloto.

Com isso esperamos ver, enquanto a corrida acontece, dois efeitos. O primeiro é o desgaste do pneu, que deixa as voltas um pouco mais lentas a cada volta. O segundo é a parada nos boxes, que gera uma volta muito lenta seguida de voltas mais rápidas com pneus novos. Essa análise está ligada à pergunta P5 do RA1.

## 9. Fonte de dados e processamento

- Informe o cenário escolhido: dados reais em streaming ou simulação.
- Explique brevemente como os novos registros chegam.
- Implemente o processamento em tempo real.
- Demonstre o recebimento de novos registros.

**Resposta do grupo:**

Escolhemos o Cenário A, de simulação. Não temos acesso a dados ao vivo de uma corrida, então criamos voltas artificiais a partir de dados reais do RA1.

As voltas simuladas são divididas em 4 lotes, e cada lote vira um arquivo. Para imitar a chegada dos dados, copiamos um arquivo de cada vez para uma pasta de entrada. O Spark Structured Streaming fica observando essa pasta e processa cada arquivo novo assim que ele aparece.

### 9.1 Base real vinda do RA1

Para a simulação ficar coerente com o RA1, usamos os 5 primeiros colocados da última corrida de 2025. Como tempo de referência, usamos a mediana das voltas normais dessa corrida, sem as voltas marcadas como atípicas. Os dados vêm dos arquivos gerados no RA1.

In [58]:
# 9.1 Pilotos e tempo-base a partir do RA1
resultados_ra1 = pd.read_csv(PASTA_PROCESSED / "resultados_limpo.csv")
voltas_ra1 = pd.read_csv(PASTA_PROCESSED / "voltas_limpo.csv")
corridas_2025 = resultados_ra1[resultados_ra1["year"] == 2025]
ultima_corrida = corridas_2025.sort_values("round")["raceId"].iloc[-1]
corrida_base = resultados_ra1[resultados_ra1["raceId"] == ultima_corrida]
pilotos_sim = corrida_base.sort_values("positionOrder").head(5)[["piloto", "equipe"]].reset_index(drop=True)
voltas_base = voltas_ra1[(voltas_ra1["raceId"] == ultima_corrida) & (~voltas_ra1["volta_atipica"])]
tempo_base = voltas_base["segundos"].median()
print("Corrida usada:", corrida_base["corrida"].iloc[0], corrida_base["year"].iloc[0])
print("Tempo-base de volta (s):", round(tempo_base, 3))
pilotos_sim

Corrida usada: Abu Dhabi Grand Prix 2025
Tempo-base de volta (s): 89.653


,piloto,equipe
0,Max Verstappen,Red Bull
1,Oscar Piastri,McLaren
2,Lando Norris,McLaren
3,Charles Leclerc,Ferrari
4,George Russell,Mercedes


### 9.2 Geração das voltas simuladas

Cada piloto corre 20 voltas. O tempo de cada volta é calculado assim:

`tempo = tempo_base + diferença_do_piloto + 0,08 × voltas_desde_o_pit + ruído`

- tempo_base é o tempo de referência da corrida real.
- diferença_do_piloto deixa cada piloto um pouco mais lento que o anterior: 0; 0,2; 0,4; 0,6 e 0,8 segundo, do 1º ao 5º colocado.
- Os 0,08 segundo por volta representam o desgaste do pneu. Esse contador volta a zero depois da parada nos boxes.
- A parada acontece numa volta sorteada entre a 9 e a 12 e soma 20 segundos naquela volta.
- O ruído é uma pequena variação aleatória, de cerca de 0,3 segundo, para os tempos não ficarem perfeitos demais. A semente 42 garante sempre o mesmo sorteio.

O horário de cada volta é o horário de início da corrida, 14h, somado aos tempos das voltas anteriores do piloto.

In [59]:
# 9.2 Geração das voltas simuladas
rng = np.random.default_rng(42)
INICIO_CORRIDA = pd.Timestamp("2025-01-01 14:00:00")
registros = []
volta_do_pit = {}
for posicao, linha in pilotos_sim.iterrows():
    volta_do_pit[linha["piloto"]] = int(rng.integers(9, 13))
    horario = INICIO_CORRIDA
    voltas_desde_pit = 0
    for volta in range(1, 21):
        tempo = tempo_base + 0.2 * posicao + 0.08 * voltas_desde_pit + rng.normal(0, 0.3)
        voltas_desde_pit = voltas_desde_pit + 1
        if volta == volta_do_pit[linha["piloto"]]:
            tempo = tempo + 20
            voltas_desde_pit = 0
        horario = horario + pd.Timedelta(seconds=tempo)
        registros.append({"piloto": linha["piloto"], "equipe": linha["equipe"], "volta": volta, "tempo_volta": round(tempo, 3), "horario": horario.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]})
voltas_simuladas = pd.DataFrame(registros)
print("Volta do pit stop de cada piloto:", volta_do_pit)
voltas_simuladas.head(6)

Volta do pit stop de cada piloto: {'Max Verstappen': 9, 'Oscar Piastri': 12, 'Lando Norris': 9, 'Charles Leclerc': 12, 'George Russell': 10}


,piloto,equipe,volta,tempo_volta,horario
0,Max Verstappen,Red Bull,1,89.341,2025-01-01 14:01:29.341
1,Max Verstappen,Red Bull,2,89.958,2025-01-01 14:02:59.299
2,Max Verstappen,Red Bull,3,90.095,2025-01-01 14:04:29.394
3,Max Verstappen,Red Bull,4,89.308,2025-01-01 14:05:58.701
4,Max Verstappen,Red Bull,5,89.582,2025-01-01 14:07:28.284
5,Max Verstappen,Red Bull,6,90.091,2025-01-01 14:08:58.375


### 9.3 Divisão em lotes

Dividimos as voltas em 4 lotes: voltas 1 a 5, 6 a 10, 11 a 15 e 16 a 20. Cada lote vira um arquivo JSON com um registro por linha. Os lotes ficam guardados na pasta data/stream/lotes e serão entregues ao Spark um de cada vez.

In [60]:
# 9.3 Gravação dos lotes em JSON
PASTA_LOTES = PASTA_STREAM / "lotes"
PASTA_ENTRADA = PASTA_STREAM / "entrada"
shutil.rmtree(PASTA_LOTES, ignore_errors=True)
PASTA_LOTES.mkdir(parents=True)
for numero in range(1, 5):
    primeira_volta = (numero - 1) * 5 + 1
    lote = voltas_simuladas[voltas_simuladas["volta"].between(primeira_volta, primeira_volta + 4)]
    lote.to_json(PASTA_LOTES / f"lote_{numero:02d}.json", orient="records", lines=True, force_ascii=False)
    print(f"lote_{numero:02d}.json: voltas {primeira_volta}-{primeira_volta + 4}, {len(lote)} registros")

lote_01.json: voltas 1-5, 25 registros
lote_02.json: voltas 6-10, 25 registros
lote_03.json: voltas 11-15, 25 registros
lote_04.json: voltas 16-20, 25 registros


### 9.4 Sessão Spark

Iniciamos o Spark em modo local, ou seja, rodando em uma só máquina, a do Colab, com todos os núcleos disponíveis. Como o volume de dados é pequeno, reduzimos para 4 o número de partes em que o Spark divide os agrupamentos.

In [61]:
# 9.4 Sessão Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, window
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType, TimestampType

construtor = SparkSession.builder.appName("F1Streaming").master("local[*]")
construtor = construtor.config("spark.ui.showConsoleProgress", "false")
spark = construtor.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")
print("Spark", spark.version)

Spark 3.5.3


### 9.5 Pasta de entrada e esquema

Limpamos a pasta de entrada, que é a pasta que o Spark vai observar, para que cada execução comece do zero. Também definimos o esquema dos registros, que é a lista de campos com o tipo de cada um. Em streaming, o Spark precisa saber isso antes de os dados chegarem.

In [62]:
# 9.5 Pasta de entrada e esquema
shutil.rmtree(PASTA_ENTRADA, ignore_errors=True)
PASTA_ENTRADA.mkdir(parents=True)
schema = StructType([
    StructField("piloto", StringType()),
    StructField("equipe", StringType()),
    StructField("volta", IntegerType()),
    StructField("tempo_volta", DoubleType()),
    StructField("horario", TimestampType()),
])
print("Arquivos na entrada:", len(list(PASTA_ENTRADA.iterdir())))

Arquivos na entrada: 0


### 9.6 Leitura em streaming

Criamos a leitura em streaming. A partir daqui o Spark sabe que deve observar a pasta de entrada e processar cada arquivo novo que aparecer, um arquivo por vez.

In [63]:
# 9.6 Leitura em streaming
fluxo = spark.readStream.schema(schema).option("maxFilesPerTrigger", 1).json(str(PASTA_ENTRADA))
print("É streaming?", fluxo.isStreaming)

É streaming? True


## 10. Aplicação de janelas temporais

- Utilize uma janela **Tumbling** (intervalos sem sobreposição) ou **Sliding** (intervalos que podem se sobrepor).
- Informe o tipo e a duração da janela e o deslocamento, quando aplicável.
- Implemente uma agregação coerente com a análise, como contagem, soma ou média.
- Explique brevemente o que será calculado em cada intervalo.

O foco é demonstrar a compreensão do conceito; não é necessário justificar tecnicamente a duração escolhida.

**Resposta do grupo:**

- **Tipo:** Tumbling, com janelas fixas, uma depois da outra, sem sobreposição. Cada volta entra em uma única janela.
- **Duração:** 5 minutos, contados pelo horário em que a volta aconteceu, e não pelo horário em que o dado chegou ao Spark.
- **Deslocamento:** não se aplica. Na janela Tumbling uma janela começa exatamente quando a anterior termina.
- **O que é calculado:** para cada janela e cada piloto, o tempo médio de volta e a quantidade de voltas.

### 10.1 Janela de 5 minutos e agregação

Agrupamos as voltas por janela de 5 minutos e por piloto. Para cada grupo calculamos a média do tempo de volta, na coluna media_volta, e o número de voltas, na coluna voltas.

In [64]:
# 10.1 Janela de 5 minutos e agregação
ritmo = fluxo.groupBy(window("horario", "5 minutes"), "piloto").agg(
    avg("tempo_volta").alias("media_volta"),
    count("volta").alias("voltas"),
)
ritmo.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- piloto: string (nullable = true)
 |-- media_volta: double (nullable = true)
 |-- voltas: long (nullable = false)



### 10.2 Início da consulta

O resultado vai para uma tabela em memória chamada ritmo. Usamos o modo complete, em que o Spark reescreve a tabela inteira a cada lote, o que deixa fácil ver as janelas mudando. Não usamos watermark, que é o limite de atraso aceito para os dados, porque com poucos dados e no modo complete ele não é necessário. A partir do start, o Spark começa de fato a observar a pasta.

In [65]:
# 10.2 Início da consulta em streaming
escrita = ritmo.writeStream.format("memory").queryName("ritmo")
consulta = escrita.outputMode("complete").start()
print("Consulta ativa:", consulta.isActive)

Consulta ativa: True


## 11. Resultados e interpretação

Apresente os resultados do processamento e demonstre como a análise se comporta com a chegada de novos registros.

### 11.1 Chegada dos lotes

Aqui simulamos a chegada dos dados. Copiamos um lote por vez para a pasta de entrada, esperamos o Spark processar e mostramos a tabela de resultado. São 4 tabelas, uma depois de cada lote.

In [66]:
# 11.1 Chegada dos lotes e resultado após cada um
CONSULTA_RITMO = """
SELECT date_format(window.start, 'HH:mm') AS inicio, date_format(window.end, 'HH:mm') AS fim,
       piloto, ROUND(media_volta, 3) AS media_volta, voltas
FROM ritmo ORDER BY inicio, piloto
"""
for numero in range(1, 5):
    shutil.copy(PASTA_LOTES / f"lote_{numero:02d}.json", PASTA_ENTRADA / f"lote_{numero:02d}.json")
    consulta.processAllAvailable()
    print(f"Após lote {numero}:")
    spark.sql(CONSULTA_RITMO).show(50, truncate=False)

Após lote 1:
+------+-----+---------------+-----------+------+
|inicio|fim  |piloto         |media_volta|voltas|
+------+-----+---------------+-----------+------+
|14:00 |14:05|Charles Leclerc|90.148     |3     |
|14:00 |14:05|George Russell |90.392     |3     |
|14:00 |14:05|Lando Norris   |90.004     |3     |
|14:00 |14:05|Max Verstappen |89.798     |3     |
|14:00 |14:05|Oscar Piastri  |89.972     |3     |
|14:05 |14:10|Charles Leclerc|90.727     |2     |
|14:05 |14:10|George Russell |90.627     |2     |
|14:05 |14:10|Lando Norris   |90.386     |2     |
|14:05 |14:10|Max Verstappen |89.445     |2     |
|14:05 |14:10|Oscar Piastri  |90.016     |2     |
+------+-----+---------------+-----------+------+

Após lote 2:
+------+-----+---------------+-----------+------+
|inicio|fim  |piloto         |media_volta|voltas|
+------+-----+---------------+-----------+------+
|14:00 |14:05|Charles Leclerc|90.148     |3     |
|14:00 |14:05|George Russell |90.392     |3     |
|14:00 |14:05|Lando Nor

### 11.2 Tabela final

Paramos a consulta e organizamos o resultado final em uma tabela com um piloto por linha e uma janela por coluna, que é mais fácil de ler.

In [67]:
# 11.2 Tabela final piloto x janela
consulta.stop()
resultado_final = spark.sql(CONSULTA_RITMO).toPandas()
resultado_final["janela"] = resultado_final["inicio"] + "–" + resultado_final["fim"]
resultado_final.pivot(index="piloto", columns="janela", values="media_volta")

janela,14:00–14:05,14:05–14:10,14:10–14:15,14:15–14:20,14:20–14:25,14:25–14:30,14:30–14:35
piloto,,,,,,,
Charles Leclerc,90.148,90.782,90.817,95.589,90.546,90.689,90.720
George Russell,90.392,90.655,91.107,95.698,90.327,90.863,91.060
Lando Norris,90.004,90.430,97.457,90.138,90.310,90.728,91.143
Max Verstappen,89.798,89.660,96.761,89.987,90.051,90.280,90.398
Oscar Piastri,89.972,90.148,90.534,95.483,90.176,90.152,90.636


### 11.3 Encerramento

Encerramos o Spark para liberar a memória.

In [68]:
# 11.3 Encerramento do Spark
spark.stop()
print("Consulta ativa:", consulta.isActive)

Consulta ativa: False


**Resposta do grupo:**

A corrida usada como base foi o GP de Abu Dhabi de 2025, com tempo de referência de 89,653 segundos. As paradas sorteadas caíram na volta 9 para Verstappen e Norris, na 10 para Russell e na 12 para Piastri e Leclerc.

- **Desgaste do pneu:** antes da parada, a média sobe de uma janela para a seguinte. Leclerc foi de 90,148 para 90,782 e depois 90,817 segundos. Russell foi de 90,392 para 90,655 e depois 91,107. Piastri foi de 89,972 para 90,148 e depois 90,534.
- **Janela da parada:** a janela que contém a volta da parada fica bem mais lenta, por causa dos 20 segundos a mais. Norris e Verstappen sobem para 97,457 e 96,761 segundos na janela das 14h10 às 14h15. Leclerc, Russell e Piastri ficam perto de 95,5 segundos na janela das 14h15 às 14h20.
- **Depois da parada:** com pneus novos, a média cai. Norris passa de 97,457 para 90,138 segundos, e Leclerc de 95,589 para 90,546. Depois o desgaste recomeça, e Norris vai para 90,310, 90,728 e 91,143.
- **Ordem dos pilotos:** Verstappen, que não tem diferença somada ao tempo, tem a menor média na maioria das janelas. O ruído às vezes inverte a ordem de pilotos com tempos próximos.
- **Chegada de novos registros:** cada lote cria janelas novas e também atualiza janelas que já existiam. Depois do lote 2, a janela das 14h15 às 14h20 de Russell tinha só 1 volta, a da parada, com 111,373 segundos. Depois do lote 3, essa mesma janela passou a ter 4 voltas e média de 95,698 segundos. A janela das 14h20 às 14h25 também foi atualizada no lote 4, de 2 para 3 voltas. A última janela, das 14h30 às 14h35, tem só 1 volta por piloto porque a corrida simulada acaba nela.

## 12. Considerações do RA2

Responda brevemente:

- O que foi possível observar com o processamento em streaming?
- Qual foi a contribuição das janelas temporais para a análise?
- Essa abordagem faria sentido em uma aplicação real do projeto? Por quê?

Na última resposta, o grupo pode reconhecer que o streaming foi utilizado apenas como exercício didático, se esse for o caso.

**Resposta do grupo:**

**O que foi possível observar:** o resultado foi sendo atualizado conforme os lotes chegavam, sem precisar processar tudo de novo. Deu para ver o desgaste do pneu, o efeito da parada nos boxes e janelas que ficaram pela metade em um lote sendo completadas no lote seguinte.

**Contribuição das janelas:** a janela de 5 minutos transforma uma sequência de voltas soltas em um resumo fácil de ler. Comparando janelas seguidas, dá para ver a tendência do ritmo de cada piloto ao longo da corrida, algo difícil de perceber volta a volta por causa da variação natural dos tempos.

**Faria sentido numa aplicação real?** Sim. Equipes e transmissões de TV acompanham o ritmo dos pilotos ao vivo para decidir a hora de parar nos boxes e para explicar a corrida ao público. Neste projeto, porém, o streaming foi um exercício didático, porque os dados são simulados com uma fórmula simples. Numa aplicação real, a fonte seria a cronometragem oficial ao vivo, e seria preciso usar watermark para lidar com dados que chegam atrasados.